# Streaming Lars Forward Return

Build LOB features from MBO data, add a forward-return target, then run rolling time-series validation with the streaming `Lars(/Ridge)Adapter`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import polars as pl

from tools.data import DateFrame, RAW_PATH, Raw
from tools.features import LOBFeatures, add_features, depth_meta
from tools.filters import intraday_time, level_taken, tight_spread, trade_size
from tools.model import LarsAdapter, RidgeAdapter
from tools.pipeline import Pipeline, get_unit_pnl, rmse
from tools.price import add_return
from tools.search import loguniform, grid
from tools.transform import Standardizer


In [2]:
from tools.data import expand_dates

expand_dates('20260501-20260507')

['2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07']

In [3]:
def divide_dates(*args):
    dates = []
    for i in range(1, len(args)):
        dates.append(expand_dates(f'{args[i - 1]}-{args[i]}', end_date=False if i < len(args) - 1 else True))
    return dates

In [4]:
# divide_dates(20260323, 20260410, 20260425, 20260510, 20260529)

In [5]:
# Data
PROD = "ES"
# ROLLING_DATES = [['2026-05-01', '2026-05-04', '2026-05-05'], ["2026-05-06"], ["2026-05-07"]]
ROLLING_DATES = divide_dates(20260323, 20260410, 20260425, 20260510, 20260529)
TEST_DATES = ["2026-05-06"]
L2_DEPTH = 5
MODEL_BATCH_SIZE = 20_000
POLARS_ENGINE = "gpu"
ORDERBOOK_PATH = str(ROOT / f"data/orderbook_l2_parquet/{{prod}}M6_{{d}}_{{tag}}_{{prod_s}}_full_day_l2_d{L2_DEPTH}.parquet")
RAW_CONTEXT_COLS = ("ts_event", "ts_recv", "symbol")
REGULAR_HOURS_START = "09:30"
REGULAR_HOURS_END = "16:00"
REGULAR_HOURS_TZ = "America/New_York"

# Feature knobs
IMBALANCE_DEPTHS = [1, 3, 5]
IMBALANCE_LOG = True
TRADE_MOMENTUM_LOG = True
WEIGHTED_PRICE_DEPTH = 5
WEIGHTED_PRICE_SIZES = [2, 5, 10]
TRADE_MOMENTUM_HALF_LIVES = [1, 10, 30, 120]  # seconds

# Forward-return target knobs
FUTURE_HORIZONS = ["10s", "30s", "60s"]
FUTURE_WEIGHTS = [0.5, 0.3, 0.2]
TARGET = "forward_mid_return_bps"
TEST_PNL_THRESHOLD = 0.3

# Lars/search knobs
SAMPLER = "random"
# SAMPLER = "grid"
N_TRIALS = 5
SEARCH_SPACE = {"alpha": loguniform(1e-2, 10)}
# SEARCH_SPACE = {"alpha": grid([0])}

UNDEF_PRICE = 9_223_372_036_854_775_807
assert L2_DEPTH >= max([*IMBALANCE_DEPTHS, WEIGHTED_PRICE_DEPTH])


In [6]:
def tag(x: object) -> str:
    return str(x).replace(".", "p")


def feature_exprs() -> dict[str, pl.Expr]:
    exprs = {}
    for depth in IMBALANCE_DEPTHS:
        exprs[f"imb_d{depth}"] = LOBFeatures.book_imbalance(depth, log=IMBALANCE_LOG)
    for size in sorted(set([*WEIGHTED_PRICE_SIZES])):
        exprs[f"weighted_price_sz{tag(size)}"] = LOBFeatures.size_weighted_avg_price(WEIGHTED_PRICE_DEPTH, size)
    for half_life in TRADE_MOMENTUM_HALF_LIVES:
        exprs[f"trade_momentum_hl{tag(half_life)}s"] = LOBFeatures.trade_momentum(half_life, log=TRADE_MOMENTUM_LOG)
    return exprs


FEATURE_EXPRS = feature_exprs()
FEATURES = list(FEATURE_EXPRS)
###
# FEATURES = ["imb_d1"]
###
MID = (pl.col("bid_px_0") + pl.col("ask_px_0")) / 2
META_COLS = ["ts_event", "ts_recv", "symbol", *depth_meta(L2_DEPTH)]
VALID_ROWS = (
    (pl.col("bid_px_0") != UNDEF_PRICE)
    & (pl.col("ask_px_0") != UNDEF_PRICE)
    & pl.col(TARGET).is_not_null()
    & pl.all_horizontal([pl.col(c).is_finite() for c in FEATURES])
)
REGULAR_HOURS = intraday_time(REGULAR_HOURS_START, REGULAR_HOURS_END, timezone=REGULAR_HOURS_TZ)
TIGHT_SPREAD = tight_spread(25)
VALID_REGULAR_ROWS = VALID_ROWS & REGULAR_HOURS & TIGHT_SPREAD
TRAIN_ROWS = VALID_REGULAR_ROWS & (level_taken() | trade_size(0.3))
FEATURES


['imb_d1',
 'imb_d3',
 'imb_d5',
 'weighted_price_sz2',
 'weighted_price_sz5',
 'weighted_price_sz10',
 'trade_momentum_hl1s',
 'trade_momentum_hl10s',
 'trade_momentum_hl30s',
 'trade_momentum_hl120s']

In [7]:
def raw_context(day: str, prod: str) -> pl.LazyFrame:
    raw_path, _ = Raw.resolve_path(day, prod, RAW_PATH)
    cols = [c for c in RAW_CONTEXT_COLS if c in pl.scan_parquet(raw_path).collect_schema().names()]
    return (
        pl.scan_parquet(raw_path)
        .select(cols)
        .with_row_index("row_nr")
        .with_columns(pl.col("row_nr").cast(pl.UInt64))
    )


def load_orderbook_date(day: str, prod: str = PROD) -> DateFrame:
    book_path, tag = Raw.resolve_path(day, prod, ORDERBOOK_PATH)
    order_cols = ["ts_event", "row_nr"]
    lf = (
        pl.scan_parquet(book_path)
        .join(raw_context(day, prod), on="row_nr", how="left")
        .sort(order_cols)
    )
    lf = add_features(lf, FEATURE_EXPRS)
    lf = add_return(lf, MID, FUTURE_HORIZONS, FUTURE_WEIGHTS, name=TARGET)
    lf = lf.sort(order_cols)
    cols = list(dict.fromkeys([*META_COLS, *FEATURES, TARGET]))
    return DateFrame(day, tag, lf.select(cols))


def loader(dates: list[str]) -> list[DateFrame]:
    return [load_orderbook_date(day) for day in dates]

pipeline = Pipeline(
    rolling_dates=ROLLING_DATES,
    test_dates=TEST_DATES,
    adapter=LarsAdapter(batch_size=MODEL_BATCH_SIZE, alpha_min=0.0, fit_intercept=False),
    # adapter=RidgeAdapter(batch_size=MODEL_BATCH_SIZE, fit_intercept=False, stats_backend="gpu"),
    target=TARGET,
    features=FEATURES,
    data_loader=loader,
    search_space=SEARCH_SPACE,
    val_score=rmse,
    transform=Standardizer(FEATURES),
    train_filters=(TRAIN_ROWS,),
    val_filters=(VALID_REGULAR_ROWS,),
    test_filters=(VALID_REGULAR_ROWS,),
    sampler=SAMPLER,
    n_trials=N_TRIALS,
    cache_arrays=False,
    score_direction="minimize",
    polars_engine=POLARS_ENGINE,
)
pipeline


Pipeline(rolling_dates=[['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09'], ['2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24'], ['2026-04-27', '2026-04-28', '2026-04-29', '2026-04-30', '2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08'], ['2026-05-11', '2026-05-12', '2026-05-13', '2026-05-14', '2026-05-15', '2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21', '2026-05-22', '2026-05-26', '2026-05-27', '2026-05-28', '2026-05-29']], test_dates=['2026-05-06'], adapter=LarsAdapter(alpha=1.0, method='lasso', fit_intercept=False, max_iter=500, alpha_min=0.0, eps=2.220446049250313e-16, positive=False, batch_size=20000, max_features=None, stats_backend='cpu', stats_dtype=<class 'numpy.float64'>, streaming=True, cache_path=True),

In [ ]:
from tools.data import DataSource

src = DataSource(
    dates=ROLLING_DATES[-1],
    loader=loader,
    target=TARGET,
    features=FEATURES,
    filters=(VALID_ROWS,),
    polars_engine=POLARS_ENGINE,
)

df = src.frame()[::100].collect()

In [ ]:
df

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
corr = np.zeros((len(FEATURES), len(FEATURES)))
for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        f = FEATURES[i]
        f1 = FEATURES[j]
        corr[i, j] = df.select(pl.corr(pl.col(f), pl.col(f1)))[0].item()
print(corr)
for f in FEATURES:
    print(f, df.select(pl.corr(pl.col(f), pl.col(TARGET)), pl.col(TARGET).std()))
    plt.scatter(df[f], df[TARGET])
    plt.show()

In [ ]:
train_result = pipeline.train(verbose=2)
train_result


In [ ]:
test_result = pipeline.test(get_unit_pnl(0.0))
test_result


In [ ]:
test_result = pipeline.test(get_unit_pnl(0.1))
test_result


In [ ]:
# path = pipeline.adapter.path(pipeline.model)
# {k: (v.shape if hasattr(v, "shape") else len(v)) for k, v in path.items()}
_ = plt.hist(test_result['y_pred'], bins=100)

In [ ]:
model = pipeline.get_model()
transform = pipeline.fitted_transform

if model is None or model.coef is None:
    raise RuntimeError("Run pipeline.train() before inspecting coefficients.")
if transform is None or transform.means is None or transform.stds is None:
    raise RuntimeError("Run pipeline.train() with a fitted Standardizer before inspecting real coefficients.")

features = list(pipeline.features)
raw_coef = [float(coef) for coef in model.coef]
means = [float(transform.means[feature]) for feature in features]
stds = [float(transform.stds[feature]) for feature in features]
real_coef = [coef / std for coef, std in zip(raw_coef, stds)]

raw_intercept = float(model.intercept)
real_intercept = raw_intercept - sum(
    coef * mean / std for coef, mean, std in zip(raw_coef, means, stds)
)

df_coef = pl.DataFrame(
    [
        {
            "term": "intercept",
            "raw_coef": raw_intercept,
            "real_coef": real_intercept,
            "standardizer_mean": None,
            "standardizer_std": None,
        },
        *[
            {
                "term": feature,
                "raw_coef": raw,
                "real_coef": real,
                "standardizer_mean": mean,
                "standardizer_std": std,
            }
            for feature, raw, real, mean, std in zip(features, raw_coef, real_coef, means, stds)
        ],
    ]
)


In [ ]:
df_coef